# Cross-Chip SD Prediction Model

**CRISP-DM Phase 4 — Modeling**  
Predicting the average Scanner Density across all 30 chips (`cross_chip_mean`) from waveform parameters.  
This answers: *what SD level will all chips produce for this setting?*

Use alongside the cross-chip consistency model (model 2) to get both the expected SD level and how consistent chips are with each other.  
See `documentation/why-sd-prediction.md` for why this is more fundamental than predicting std directly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False
    print('XGBoost not available — skipping.')

## 1. Load and prepare
*CRISP-DM Phase 3 — Data Preparation*

Compute `cross_chip_mean` — the mean of the 30 per-chip mean SD values.  
This is the actual SD level all chips are expected to produce together.

In [ ]:
df = pd.read_parquet('../../data/new/extended-rows.parquet 2')
value_cols = [c for c in df.columns if 'Value' in c]
cond_cols  = ['V', 'F_r', 'dt2', 'Coverage#']

df['color_idx'] = df.groupby(cond_cols).cumcount()
df['Color$']    = df['color_idx'].map({0: 'M', 1: 'C', 2: 'Y', 3: 'K'})

chip_means = pd.DataFrame(index=df.index)
for chip in range(1, 31):
    cols = [c for c in value_cols if c.startswith(f'{float(chip)}_')]
    chip_means[f'chip_{chip}'] = df[cols].mean(axis=1)

df['cross_chip_mean'] = chip_means.mean(axis=1)
df['cross_chip_std']  = chip_means.std(axis=1)

print(f'Rows: {len(df):,}')
print(f'cross_chip_mean range: {df["cross_chip_mean"].min():.5f} – {df["cross_chip_mean"].max():.5f}')

## 2. Feature engineering
*CRISP-DM Phase 3 — Data Preparation*

In [ ]:
color_dummies = pd.get_dummies(df['Color$'], prefix='color', drop_first=False)
base = df[['V', 'F_r', 'dt2', 'Coverage#']].copy()
base['V_x_Fr']         = df['V'] * df['F_r']
base['dt2_x_coverage'] = df['dt2'] * df['Coverage#']
base['V_sq']           = df['V'] ** 2
base['Fr_sq']          = df['F_r'] ** 2
X = pd.concat([base, color_dummies], axis=1)
y = df['cross_chip_mean']
print(f'Features: {X.columns.tolist()}')

## 3. Train / test split
*CRISP-DM Phase 4 — Modeling: Generate Test Design*

Random 80/20 split stratified by Color.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df['Color$'])
print(f'Train: {len(X_train):,} rows')
print(f'Test:  {len(X_test):,} rows')

## 4. Train and evaluate models
*CRISP-DM Phase 4 — Modeling: Build Model*

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}
if XGBOOST_AVAILABLE:
    models['XGBoost'] = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42, verbosity=0)

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {'R2': r2_score(y_test, y_pred), 'MAE': mean_absolute_error(y_test, y_pred), 'model': model, 'y_pred': y_pred}

summary = pd.DataFrame({k: {'R²': v['R2'], 'MAE': v['MAE']} for k, v in results.items()}).T
print(summary.round(4))

## 5. Predicted vs actual
*CRISP-DM Phase 5 — Evaluation*

In [ ]:
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
if n == 1: axes = [axes]
for ax, (name, res) in zip(axes, results.items()):
    ax.scatter(y_test, res['y_pred'], alpha=0.3, s=5)
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, 'r--', linewidth=1)
    ax.set_xlabel('Actual cross_chip_mean')
    ax.set_ylabel('Predicted cross_chip_mean')
    ax.set_title(f'{name}\nR²={res["R2"]:.3f}  MAE={res["MAE"]:.5f}')
plt.tight_layout()
plt.show()

## 6. Manual prediction
*CRISP-DM Phase 5 — Evaluation*

Change the values below. Compare with model 2 (cross_chip_std) to get both the SD level and the consistency.

In [ ]:
best_name  = max(results, key=lambda k: results[k]['R2'])
best_model = results[best_name]['model']
print(f'Best model: {best_name}  (R²={results[best_name]["R2"]:.4f})')

# --- change these values ---
V = 20.0; F_r = 1.26; dt2 = -1100; coverage = 10.0; color = 'C'
# ---------------------------

custom = pd.DataFrame([{'V': V, 'F_r': F_r, 'dt2': dt2, 'Coverage#': coverage, 'Color$': color}])
cd = pd.get_dummies(custom['Color$'], prefix='color', drop_first=False)
b = custom[['V','F_r','dt2','Coverage#']].copy()
b['V_x_Fr'] = custom['V']*custom['F_r']
b['dt2_x_coverage'] = custom['dt2']*custom['Coverage#']
b['V_sq'] = custom['V']**2
b['Fr_sq'] = custom['F_r']**2
Xc = pd.concat([b, cd], axis=1).reindex(columns=X_train.columns, fill_value=0)
pred_mean = best_model.predict(Xc)[0]

match = df[(df['V']==V)&(df['F_r']==F_r)&(df['dt2']==dt2)&(df['Coverage#']==coverage)&(df['Color$']==color)]
print(f'Settings: V={V}, F_r={F_r}, dt2={dt2}, Coverage={coverage}, Color={color}')
print(f'Predicted cross_chip_mean (SD level): {pred_mean:.6f}')
if len(match):
    print(f'Actual    cross_chip_mean:            {match["cross_chip_mean"].values[0]:.6f}')
    print(f'Actual    cross_chip_std:             {match["cross_chip_std"].values[0]:.6f}')
else:
    print('Actual: not in dataset — prediction only')